# Live Demo: Icarus Verilog + Covered

This notebook follows the slide deck. Run the cells from top to bottom while presenting. **The first code cell installs all required command-line tools.**

The example intentionally tests **ADD** and **AND**, but not **OR** or **XOR**.

In [1]:
# INSTALL CELL: run this cell before every other code cell.
# Linux/Google Colab: installs packages with apt-get.
# macOS: installs Icarus with Homebrew and builds Covered locally.

from pathlib import Path
from IPython.display import Markdown, display
import hashlib
import os
import platform
import shlex
import shutil
import subprocess
import tarfile
import urllib.request

def run_setup(command, cwd=None, env=None):
    print("$", shlex.join(str(item) for item in command))
    result = subprocess.run([str(item) for item in command], cwd=cwd, env=env, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if result.returncode != 0:
        print(result.stdout)
        raise RuntimeError(f"Setup command failed with exit code {result.returncode}")
    print("  done")

# Find the example directory.
cwd = Path.cwd().resolve()
if (cwd / "src/simple_alu.v").exists():
    ROOT = cwd
elif (cwd / "coverage_exp/src/simple_alu.v").exists():
    ROOT = cwd / "coverage_exp"
else:
    raise FileNotFoundError("Keep this notebook inside coverage_exp, or start Jupyter from its parent directory.")

os.chdir(ROOT)
(ROOT / "build").mkdir(exist_ok=True)
TOOLS = ROOT / ".tools"
system = platform.system()
# Covered 0.7.10 cannot compile when its install prefix contains spaces.
# Use a short per-user macOS path, so folders such as 'coverage_exp 2' work.
COVERED_PREFIX = (Path("/tmp") / f"coverage_exp_covered_0_7_10_{os.getuid()}"
                  if system == "Darwin" else TOOLS / "covered")
LOCAL_COVERED = COVERED_PREFIX / "bin/covered"
print("Detected system:", system)

# Jupyter launched from a GUI may not inherit the usual Homebrew PATH.
if system == "Darwin":
    os.environ["PATH"] = "/opt/homebrew/bin:/usr/local/bin:" + os.environ.get("PATH", "")
    os.environ.setdefault("HOMEBREW_NO_AUTO_UPDATE", "1")
    os.environ.setdefault("HOMEBREW_NO_INSTALL_CLEANUP", "1")

if system == "Linux":
    missing = [name for name in ("iverilog", "vvp", "covered") if not shutil.which(name)]
    if missing:
        apt_get = shutil.which("apt-get")
        if not apt_get:
            raise RuntimeError("This Linux system has no apt-get. Use the course's Ubuntu/Colab environment.")
        prefix = [] if not hasattr(os, "geteuid") or os.geteuid() == 0 else ["sudo", "-n"]
        run_setup(prefix + [apt_get, "update", "-qq"])
        run_setup(prefix + [apt_get, "install", "-y", "iverilog", "covered"])

elif system == "Darwin":
    brew = shutil.which("brew")
    if not brew:
        raise RuntimeError("Homebrew is required on macOS. Install it from https://brew.sh, restart Jupyter, and rerun this cell.")

    if not shutil.which("iverilog") or not shutil.which("vvp"):
        run_setup([brew, "install", "icarus-verilog"])

    if not shutil.which("covered") and not LOCAL_COVERED.exists():
        # Covered 0.7.10 is old, so build a private copy without changing /usr/local.
        build_formulas = ["bison", "flex", "gperf", "autoconf", "automake"]
        missing_formulas = [name for name in build_formulas if subprocess.run(
            [brew, "list", "--versions", name], stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL).returncode != 0]
        if missing_formulas:
            run_setup([brew, "install"] + missing_formulas)
        brew_prefix = subprocess.check_output([brew, "--prefix"], text=True).strip()
        build_env = os.environ.copy()
        build_env["PATH"] = f"{brew_prefix}/opt/bison/bin:{brew_prefix}/opt/flex/bin:{brew_prefix}/bin:" + build_env["PATH"]

        version = "0.7.10"
        url = "https://deb.debian.org/debian/pool/main/c/covered/covered_0.7.10.orig.tar.gz"
        expected_sha256 = "08b859665ae1351d9641ebab08d049beb378ba3bceb4ed2c5fac01b2f1738f57"
        archive = TOOLS / f"covered-{version}.tar.gz"
        build_root = TOOLS / "covered-build"
        source = build_root / f"covered-{version}"
        TOOLS.mkdir(exist_ok=True)

        if not archive.exists() or hashlib.sha256(archive.read_bytes()).hexdigest() != expected_sha256:
            print("Downloading pinned Covered source...")
            request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            archive.write_bytes(urllib.request.urlopen(request).read())
        actual_sha256 = hashlib.sha256(archive.read_bytes()).hexdigest()
        if actual_sha256 != expected_sha256:
            raise RuntimeError("Covered download checksum did not match.")

        if build_root.exists():
            shutil.rmtree(build_root)
        build_root.mkdir()
        with tarfile.open(archive, "r:gz") as package:
            try:
                package.extractall(build_root, filter="data")
            except TypeError:
                package.extractall(build_root)

        # Modern Clang needs this small compatibility change for old Covered source.
        header = source / "src/lxt2_read.h"
        header.write_text(header.read_text().replace("#define _LXT2_RD_INLINE inline", "#define _LXT2_RD_INLINE"))
        run_setup(["./configure", f"--prefix={COVERED_PREFIX}"], cwd=source, env=build_env)
        run_setup(["make", "-j2"], cwd=source, env=build_env)
        run_setup(["make", "install"], cwd=source, env=build_env)

elif system == "Windows":
    raise RuntimeError("Use this notebook in Google Colab or WSL/Ubuntu on Windows. Native Windows installation is not supported.")
else:
    raise RuntimeError(f"Unsupported operating system: {system}")

IVERILOG = shutil.which("iverilog")
VVP = shutil.which("vvp")
COVERED = (os.environ.get("COVERED") or shutil.which("covered") or
           (str(LOCAL_COVERED) if LOCAL_COVERED.exists() else None))
missing = [name for name, path in [("iverilog", IVERILOG), ("vvp", VVP), ("covered", COVERED)] if not path]
if missing:
    raise RuntimeError("Installation did not provide: " + ", ".join(missing))

def run_command(args, shown=None):
    print("$", shown or shlex.join(str(item) for item in args))
    result = subprocess.run([str(item) for item in args], cwd=ROOT, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout.rstrip() if result.stdout else "(no terminal output; command succeeded)")
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}")
    return None

print("READY: iverilog, vvp, and covered are installed.")

Detected system: Darwin
READY: iverilog, vvp, and covered are installed.


## Slide 3 — Look at the RTL

The calculator has two number inputs, one operation input, and one result output.

In [2]:
rtl_text = (ROOT / "src/simple_alu.v").read_text()
display(Markdown("```verilog\n" + rtl_text + "\n```"))

```verilog
`timescale 1ps/1ps

// A small 4-bit calculator used to demonstrate basic code coverage.
module simple_alu (
    input  wire [3:0] a,
    input  wire [3:0] b,
    input  wire [1:0] op,
    output reg  [3:0] result
);

    // op selects one of four simple operations.
    always @* begin
        case (op)
            2'b00: result = a + b;  // ADD
            2'b01: result = a & b;  // AND
            2'b10: result = a | b;  // OR
            2'b11: result = a ^ b;  // XOR
        endcase
    end

endmodule

```

## Slide 4 — Look at the testbench

Find the two tests and the comment saying that OR and XOR are not tested.

In [3]:
tb_text = (ROOT / "tb/simple_alu_tb.v").read_text()
display(Markdown("```verilog\n" + tb_text + "\n```"))

```verilog
`timescale 1ns/1ps

module simple_alu_tb;
    reg  [3:0] a;
    reg  [3:0] b;
    reg  [1:0] op;
    wire [3:0] result;

    integer failures;

    simple_alu dut (
        .a(a),
        .b(b),
        .op(op),
        .result(result)
    );

    initial begin
        $dumpfile("build/simple_alu_raw.vcd");
        $dumpvars(0, simple_alu_tb);

        failures = 0;
        a = 4'h0;
        b = 4'h0;
        op = 2'b00;

        // Test 1: ADD.  3 + 2 should be 5.
        #1;
        a = 4'h3;
        b = 4'h2;
        op = 2'b00;
        #1;
        if (result !== 4'h5) begin
            $display("ERROR: ADD result=%h expected=5", result);
            failures = failures + 1;
        end

        // Test 2: AND.  6 & 3 should be 2.
        #1;
        a = 4'h6;
        b = 4'h3;
        op = 2'b01;
        #1;
        if (result !== 4'h2) begin
            $display("ERROR: AND result=%h expected=2", result);
            failures = failures + 1;
        end

        // OR (op=10) and XOR (op=11) are intentionally not tested.
        if (failures == 0)
            $display("PASS: ADD and AND tests passed");
        else
            $display("FAIL: %0d self-check(s) failed", failures);

        #1;
        $finish;
    end

endmodule

```

## Slide 5 — Compile with Icarus Verilog

This creates the simulation program `build/simple_alu.vvp`.

In [4]:
run_command(
    [IVERILOG, "-g2001", "-Wall", "-s", "simple_alu_tb",
     "-o", "build/simple_alu.vvp", "src/simple_alu.v", "tb/simple_alu_tb.v"],
    shown="iverilog -g2001 -Wall -s simple_alu_tb -o build/simple_alu.vvp src/simple_alu.v tb/simple_alu_tb.v"
)

$ iverilog -g2001 -Wall -s simple_alu_tb -o build/simple_alu.vvp src/simple_alu.v tb/simple_alu_tb.v
(no terminal output; command succeeded)


## Slide 5 — Run the simulation

The important line is `PASS: ADD and AND tests passed`. The simulation also writes a VCD waveform.

In [5]:
run_command([VVP, "build/simple_alu.vvp"], shown="vvp build/simple_alu.vvp")

$ vvp build/simple_alu.vvp
VCD info: dumpfile build/simple_alu_raw.vcd opened for output.
PASS: ADD and AND tests passed
tb/simple_alu_tb.v:56: $finish called at 5000 (1ps)


## Slide 6 — Prepare the waveform for Covered

Covered 0.7.10 is older than Icarus Verilog 13. This removes one VCD comment that the old parser cannot read.

In [6]:
raw_vcd = (ROOT / "build/simple_alu_raw.vcd").read_text().splitlines(keepends=True)
clean_vcd = [line for line in raw_vcd if not line.startswith("$comment " )]
clean_text = "".join(clean_vcd).replace("$timescale\n\t1ps\n$end\n",
                                             "$timescale 1ps $end\n")
clean_path = ROOT / "build/simple_alu.vcd"
temporary_path = ROOT / "build/simple_alu.vcd.tmp"
temporary_path.write_text(clean_text)
temporary_path.replace(clean_path)  # Atomic replacement helps waveform viewers reload safely.
print("$ sed '/^$comment /d' build/simple_alu_raw.vcd > build/simple_alu.vcd")
print("Created build/simple_alu.vcd with a Fliplot-friendly 1ps timescale")

$ sed '/^$comment /d' build/simple_alu_raw.vcd > build/simple_alu.vcd
Created build/simple_alu.vcd with a Fliplot-friendly 1ps timescale


## Slides 6–7 — Score coverage and create the CDD

`covered score` reads the RTL and the VCD waveform, then creates a coverage database (`.cdd`). The CDD saves the design structure and the measured hit/miss data. `covered report` reads this CDD in the next step.

In [7]:
run_command(
    [COVERED, "score", "-t", "simple_alu", "-i", "simple_alu_tb.dut",
     "-v", "src/simple_alu.v", "-vcd", "build/simple_alu.vcd",
     "-o", "build/simple_alu.cdd"],
    shown="covered score -t simple_alu -i simple_alu_tb.dut -v src/simple_alu.v -vcd build/simple_alu.vcd -o build/simple_alu.cdd"
)

$ covered score -t simple_alu -i simple_alu_tb.dut -v src/simple_alu.v -vcd build/simple_alu.vcd -o build/simple_alu.cdd

Covered covered-0.7.10 -- Verilog Code Coverage Utility
Written by Trevor Williams  (phase1geo@gmail.com)
Freely distributable under the GPL license

Reading design...
Parsing file 'src/simple_alu.v'

Checking for race conditions...

Scoring VCD dumpfile build/simple_alu.vcd...

***  Scoring completed successfully!  ***

Dynamic memory allocated:   50182 bytes


## Slide 6 — Generate two reports from the CDD

The summary gives percentages. The detailed report gives missed RTL lines.

In [8]:
run_command(
    [COVERED, "report", "-m", "ltc", "-d", "s", "-o",
     "build/coverage_summary.rpt", "build/simple_alu.cdd"],
    shown="covered report -m ltc -d s -o build/coverage_summary.rpt build/simple_alu.cdd"
)
run_command(
    [COVERED, "report", "-m", "ltc", "-d", "v", "-i", "-o",
     "build/coverage_detail.rpt", "build/simple_alu.cdd"],
    shown="covered report -m ltc -d v -i -o build/coverage_detail.rpt build/simple_alu.cdd"
)

$ covered report -m ltc -d s -o build/coverage_summary.rpt build/simple_alu.cdd

Covered covered-0.7.10 -- Verilog Code Coverage Utility
Written by Trevor Williams  (phase1geo@gmail.com)
Freely distributable under the GPL license
$ covered report -m ltc -d v -i -o build/coverage_detail.rpt build/simple_alu.cdd

Covered covered-0.7.10 -- Verilog Code Coverage Utility
Written by Trevor Williams  (phase1geo@gmail.com)
Freely distributable under the GPL license


## Slides 8–12 — Coverage examples and the real summary

The next cell prints the real `simple_alu` rows from the generated summary report.

In [9]:
summary_text = (ROOT / "build/coverage_summary.rpt").read_text()
sections = [
    ("LINE COVERAGE", "LINE COVERAGE RESULTS"),
    ("TOGGLE COVERAGE", "TOGGLE COVERAGE RESULTS"),
    ("LOGIC COVERAGE", "COMBINATIONAL LOGIC COVERAGE RESULTS"),
]
summary_lines = summary_text.splitlines()
for short_name, report_name in sections:
    start = next(i for i, line in enumerate(summary_lines) if report_name in line)
    row = next(line for line in summary_lines[start:] if line.strip().startswith("simple_alu"))
    print(short_name)
    print(row)
    print()

LINE COVERAGE
  simple_alu              simple_alu.v               3/    2/    5       60%

TOGGLE COVERAGE
  simple_alu              simple_alu.v               9/    5/   14       64%             3/   11/   14       21%

LOGIC COVERAGE
  simple_alu                        simple_alu.v                        6/  11/  17       35%



## Slide 12 — Show the missed RTL lines

This is the clearest result: the test passed, but two operations were not tested.

In [10]:
detail_text = (ROOT / "build/coverage_detail.rpt").read_text()
start = detail_text.index("Missed Lines")
end = detail_text.index("~~~", start)
print(detail_text[start:end].rstrip())

Missed Lines

           16:    result =  a  |  b 
           17:    result =  a  ^  b


## Slide 13 — What should we add next?

Add these two test cases to the Verilog testbench, then run the notebook again:

```verilog
// OR test
a = 4'h6; b = 4'h3; op = 2'b10;  // expected result: 7

// XOR test
a = 4'h6; b = 4'h3; op = 2'b11;  // expected result: 5
```

The line coverage should improve because the OR and XOR lines will run.